# 🎯 Module 16: Model Deployment

> **Stage:** Deep Learning  
> **Level:** Intermediate → Job-Ready  
> **Module:** Model Deployment  
> **Framework:** PyTorch + FastAPI + Docker  
> **Date:** 16 June 2026  
> **Previous:** 15 — NLP with Deep Learning  
> **Next:** 17 — Deep Learning Projects

---

Training a model is only **half the story**. A model that never reaches users creates zero value.

**Deployment** turns a trained model into a service that applications and users can query.

In this notebook we'll cover:

- Saving & loading models (`state_dict`)
- **Inference mode** — `model.eval()` and `torch.no_grad()`
- Why **preprocessing must match** training
- Serving with **FastAPI**
- Request / response schemas
- **Dockerizing** the model
- **CPU vs GPU** inference
- **Real-time vs batch** inference
- Health checks, error handling, logging
- **ONNX** and cloud deployment

## 📚 Learning Objectives

By the end of this notebook you should be able to:

- Save and load PyTorch models using **`state_dict`**
- Correctly switch to **inference mode**
- Reproduce **preprocessing** exactly at inference time
- Expose a model via **FastAPI**
- Design clean **request / response** schemas
- **Dockerize** a model service
- Reason about **CPU vs GPU** inference
- Distinguish **real-time** vs **batch** inference
- Implement **health checks**, **error handling**, and **logging**
- Evaluate **ONNX** and **cloud deployment** options

---
## ⚙️ 0. Setup & Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

import numpy as np
import matplotlib.pyplot as plt
import os, time, json
from pathlib import Path

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else
                      'mps'  if torch.backends.mps.is_available() else 'cpu')
print(f'Using device: {device}')
print(f'PyTorch version: {torch.__version__}')

---
## 🚀 1. What Is Model Deployment?

Training a model is only part of an AI system.

**Model deployment** means making a trained model available so an application or user can send data and receive predictions.

### The pipeline

```text
Training
   ↓
Save Model
   ↓
Package Model
   ↓
Serve Model
   ↓
API / Application
   ↓
Prediction
```

### Example: image classification

```text
User uploads image
       ↓
Frontend
       ↓
FastAPI
       ↓
PyTorch Model
       ↓
Prediction
       ↓
JSON Response
```

---
## 💾 2. Saving & Loading a PyTorch Model

The recommended approach is to save the model's **`state_dict`** — a dictionary of the learned parameters, not the code.

### Save

```python
torch.save(model.state_dict(), "model.pth")
```

### Load

```python
model = MyModel()
model.load_state_dict(torch.load("model.pth", map_location="cpu"))
model.eval()
```

### ⚠️ The model architecture must be available

A `state_dict` contains **weights only** — you still need the class definition.

### 📋 Keep track of for deployment

| Item | Why |
| --- | --- |
| Model version | Reproducibility |
| Input shape | Correct tensor shapes |
| Preprocessing | Consistency with training |
| Class names | Human-readable output |
| Framework version | Compatibility |
| Required dependencies | Reproducible environment |

In [ ]:
# A simple demo model to demonstrate save / load
class DemoNet(nn.Module):
    def __init__(self, in_dim=10, hidden=32, out_dim=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, out_dim),
        )
    def forward(self, x):
        return self.net(x)

model = DemoNet()

# Save state_dict
os.makedirs('checkpoints', exist_ok=True)
path = 'checkpoints/demo_model.pth'
torch.save(model.state_dict(), path)
print(f'Saved to {path}')
print(f'File size: {os.path.getsize(path) / 1024:.1f} KB')

# Load into a new instance
model2 = DemoNet()
model2.load_state_dict(torch.load(path, map_location='cpu'))
model2.eval()
print('\nModel loaded successfully.')

---
## 🧊 3. Inference Mode

Training and inference behave **differently**.

### Correct inference pattern

```python
model.eval()
with torch.no_grad():
    prediction = model(x)
```

### What these do

| Call | Effect |
| --- | --- |
| `model.eval()` | Dropout off, BatchNorm uses running stats |
| `torch.no_grad()` | No gradient graph built → less memory, faster |

⚠️ Forgetting either one is one of the most common deployment bugs.

In [ ]:
# Demonstrate the difference: train mode vs eval mode
class DropoutNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(8, 8)
        self.drop = nn.Dropout(0.5)
    def forward(self, x):
        return self.drop(self.fc(x))

m = DropoutNet()
x = torch.ones(1, 8)

m.train()
print('TRAIN mode (dropout active):')
for _ in range(3):
    print('  ', m(x).detach().numpy().round(3))

m.eval()
print('\nEVAL mode (dropout disabled):')
for _ in range(3):
    print('  ', m(x).detach().numpy().round(3))

# Show no_grad saves memory
m.eval()
with torch.no_grad():
    out = m(x)
print('\nout.requires_grad:', out.requires_grad)

---
## 🔧 4. Preprocessing Must Match

One of the **most common deployment mistakes** is using different preprocessing during training and production.

### Training

```text
Resize → Normalize → Tensor → Model
```

### Production must use the **same** pipeline

```text
User Input → Same Resize → Same Normalize → Tensor → Model
```

⚠️ If preprocessing changes, predictions can become **unreliable** even when the model itself is correct.

### Best practice

Bundle **preprocessing code** with the model so they are versioned and shipped together.

In [ ]:
# A canonical preprocessing class bundled with the model
class Preprocessor:
    def __init__(self, mean, std):
        self.mean = np.array(mean)
        self.std  = np.array(std)

    def __call__(self, x):
        # x: numpy array already resized to model input shape
        x = (x - self.mean) / self.std
        return torch.tensor(x, dtype=torch.float32).unsqueeze(0)

pre = Preprocessor(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
raw = np.random.rand(3, 224, 224).astype(np.float32)
tensor = pre(raw)
print('Input numpy :', raw.shape)
print('Output torch:', tensor.shape, ' dtype:', tensor.dtype)

---
## ⚡ 5. Serving a Model with FastAPI

[FastAPI](https://fastapi.tiangolo.com/) is a modern, fast Python web framework. It's the go-to for exposing ML models via HTTP.

### Basic service

```python
from fastapi import FastAPI
import torch

app = FastAPI()

model = MyModel()
model.load_state_dict(torch.load("model.pth", map_location="cpu"))
model.eval()

@app.post("/predict")
def predict(data: InputData):
    x = preprocess(data)
    with torch.no_grad():
        output = model(x)
    prediction = output.argmax(dim=1).item()
    return {"prediction": prediction}
```

### The architecture

```text
Frontend
   ↓ HTTP
FastAPI
   ↓
Preprocessing
   ↓
PyTorch
   ↓
Prediction
   ↓
JSON
```

Install with:
```bash
pip install fastapi uvicorn
```

In [ ]:
# Write a complete FastAPI app to disk (for reference)
app_code = '''
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import torch
import torch.nn as nn
import numpy as np

# ---- Define model (same architecture as training) ----
class DemoNet(nn.Module):
    def __init__(self, in_dim=10, hidden=32, out_dim=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, out_dim),
        )
    def forward(self, x):
        return self.net(x)

# ---- Request / Response schemas ----
class PredictRequest(BaseModel):
    features: list[float]          # length 10

class PredictResponse(BaseModel):
    predicted_class: int
    confidence: float
    model_version: str

# ---- App + model loading ----
app = FastAPI(title="Demo Model API", version="1.0.0")
MODEL_VERSION = "1.0.0"

model = DemoNet()
model.load_state_dict(torch.load("checkpoints/demo_model.pth", map_location="cpu"))
model.eval()

# ---- Endpoints ----
@app.get("/health")
def health():
    return {"status": "ok", "model_version": MODEL_VERSION}

@app.post("/predict", response_model=PredictResponse)
def predict(req: PredictRequest):
    if len(req.features) != 10:
        raise HTTPException(status_code=400, detail="features must be length 10")
    x = torch.tensor(req.features, dtype=torch.float32).unsqueeze(0)
    with torch.no_grad():
        logits = model(x)
        probs = torch.softmax(logits, dim=1)[0]
        pred = int(probs.argmax().item())
        conf = float(probs[pred].item())
    return PredictResponse(predicted_class=pred, confidence=conf, model_version=MODEL_VERSION)
'''

os.makedirs('deploy', exist_ok=True)
with open('deploy/main.py', 'w') as f:
    f.write(app_code)

print('Wrote deploy/main.py')
print('Run with:  uvicorn deploy.main:app --reload')

---
## 📋 6. Request & Response Schemas

A production API should **clearly define** its input and output.

### Text model request

```json
{
  "text": "This product is excellent"
}
```

### Response

```json
{
  "label": "positive",
  "confidence": 0.94,
  "model_version": "1.0.0"
}
```

### Why schemas matter

- **Validation** catches malformed input before the model runs
- **Documentation** is auto-generated (Swagger / OpenAPI)
- **Consistency** across clients

For image models, the endpoint receives an **uploaded file** rather than JSON.

---
## 🐳 7. Dockerizing the Model

Docker packages the application and its dependencies into a **reproducible environment**.

### Project structure

```text
project/
├── app/
│   ├── main.py
│   ├── model.py
│   └── preprocessing.py
├── model.pth
├── requirements.txt
└── Dockerfile
```

### Dockerfile

```dockerfile
FROM python:3.12-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY . .

CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]
```

Now the same application runs **consistently** across development and deployment environments.

In [ ]:
# Write a Dockerfile + requirements file for reference
dockerfile = '''\
FROM python:3.12-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY . .

# Expose the API port
EXPOSE 8000

CMD ["uvicorn", "deploy.main:app", "--host", "0.0.0.0", "--port", "8000"]
'''

requirements = '''\
torch>=2.0
fastapi>=0.110
uvicorn[standard]>=0.27
pydantic>=2.0
numpy
'''

with open('deploy/Dockerfile', 'w') as f:
    f.write(dockerfile)
with open('deploy/requirements.txt', 'w') as f:
    f.write(requirements)

print('Wrote deploy/Dockerfile and deploy/requirements.txt')
print('\nBuild & run:')
print('  docker build -t demo-api .')
print('  docker run -p 8000:8000 demo-api')

---
## 🖥️ 8. CPU vs GPU Inference

Small models often run fine on **CPU**.
Large deep-learning models benefit significantly from **GPUs**.

### Device logic

```python
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
x = x.to(device)
with torch.no_grad():
    output = model(x)
```

### What determines the choice

| Factor | CPU | GPU |
| --- | --- | --- |
| Small models | ✅ Fine | Overkill |
| Large models | 🐢 Slow | ✅ Fast |
| Cost | Low | Higher |
| Latency | OK | Excellent |
| Availability | Everywhere | Cloud only |

⚠️ **Device mismatch** is a common bug: model on GPU, input on CPU (or vice versa).

---
## ⏱️ 9. Real-Time vs Batch Inference

### Real-Time

Prediction happens **immediately** after a request:

```text
Request → Model → Response
```

Useful for:
- Chat applications
- Image classification APIs
- Fraud checks
- Recommendation requests

### Batch

Process **many samples together**:

```text
1000 inputs
    ↓
Batch processing
    ↓
Predictions
```

Useful for:
- Daily reports
- Large datasets
- Offline document processing

**Batching improves hardware utilization** — often dramatically.

In [ ]:
# Compare real-time vs batch inference on the demo model
m = DemoNet().eval()
single_input = torch.randn(1, 10)
batch_input  = torch.randn(256, 10)

def time_inference(model, x, iters=200):
    model.eval()
    with torch.no_grad():
        # warmup
        for _ in range(5):
            _ = model(x)
        t0 = time.time()
        for _ in range(iters):
            _ = model(x)
    return (time.time() - t0) / iters

t_single = time_inference(m, single_input, iters=200)
t_batch  = time_inference(m, batch_input,  iters=50)

print(f'Single-item inference: {t_single*1000:.3f} ms')
print(f'Batch (256) inference: {t_batch*1000:.3f} ms')
print(f'Per-item cost in batch: {t_batch/256*1000:.4f} ms  →  {t_single/(t_batch/256):.1f}x speedup')

---
## ❤️ 10. Health Checks & Error Handling

A production API should provide a **health endpoint**:

```python
@app.get("/health")
def health():
    return {"status": "ok"}
```

### Also handle

- Invalid input
- Missing files
- Model-loading errors
- Unexpected exceptions
- Excessively large requests
- Timeouts

### ⚠️ Security

Do **not** expose internal stack traces or sensitive information to users.
Return generic error messages and log details internally.

---
## 🏗️ 11. Production Structure

A clean production architecture:

```text
Client
  ↓
API
  ↓
Validation
  ↓
Preprocessing
  ↓
Model
  ↓
Postprocessing
  ↓
Response

Supporting:
├── Logging
├── Health checks
├── Error handling
└── Monitoring
```

### Logging rules

Log useful operational information:
- Request timing
- Errors
- Model version
- Request metadata

**Do not** log sensitive user data.

---
## 📦 12. ONNX and Other Export Formats

**ONNX (Open Neural Network Exchange)** is a cross-framework standard for model export.

### Potential benefits

- Cross-framework / runtime compatibility
- Optimized inference
- Deployment flexibility (ONNX Runtime, TensorRT, etc.)

### ⚠️ Caveats

Exporting is **not automatically better** than native PyTorch. **Test accuracy and performance** before switching runtimes.

### Example export

```python
torch.onnx.export(
    model,
    dummy_input,
    "model.onnx",
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={'input': {0: 'batch'}, 'output': {0: 'batch'}},
)
```

In [ ]:
# Try exporting the demo model to ONNX (safe to skip if onnx not installed)
try:
    m = DemoNet().eval()
    dummy = torch.randn(1, 10)
    torch.onnx.export(
        m, dummy, 'checkpoints/demo_model.onnx',
        input_names=['input'], output_names=['output'],
        dynamic_axes={'input': {0: 'batch'}, 'output': {0: 'batch'}},
        opset_version=17,
    )
    print('ONNX exported:', 'checkpoints/demo_model.onnx')
    print('Size:', os.path.getsize('checkpoints/demo_model.onnx') / 1024, 'KB')
except Exception as e:
    print('ONNX export skipped (install onnx to enable):', e)

---
## ☁️ 13. Cloud Deployment

A basic cloud deployment:

```text
GitHub
   ↓
Docker Image
   ↓
Cloud Server
   ↓
FastAPI
   ↓
PyTorch Model
```

### Possible infrastructure

- **CPU server** for smaller models
- **GPU server** for heavy models
- **Object storage** for model files
- **Managed database** for application data
- **Reverse proxy / load balancer**

### Common platforms

- AWS (ECS, SageMaker, EC2)
- GCP (Cloud Run, Vertex AI)
- Azure (Container Apps, ML)
- Hugging Face Spaces (for demos)
- Render / Railway / Fly.io (simple deploys)

At this stage, understand the **architecture** rather than every service.

---
## 🚨 14. Common Deployment Mistakes

### ❌ 1. Forgetting `model.eval()`
Dropout and BatchNorm behave differently at inference.

### ❌ 2. Forgetting `torch.no_grad()`
Wastes memory and CPU building the gradient graph.

### ❌ 3. Loading the model for every request
Load **once at startup**, keep it in memory.

### ❌ 4. Using different preprocessing in production
Match training preprocessing exactly.

### ❌ 5. Hardcoding secrets
Use environment variables or a secret manager.

### ❌ 6. Not validating inputs
Bad input → crashes / unpredictable behavior. Validate early.

### ❌ 7. No health endpoint
You can't monitor the service without it.

### ❌ 8. Shipping unnecessary dependencies
Bloat slows container start time and increases attack surface.

### ❌ 9. Ignoring CPU/GPU device mismatch
Model on GPU, input on CPU → silent failure or crash.

### ❌ 10. Returning raw internal errors
Security risk. Return generic errors; log details.

### ❌ 11. Having no model version info
Cannot reproduce or roll back. Tag every model.

---
## 🏋️ 15. Hands-On Exercises

### Exercise 1 — Build a Real FastAPI Service
Take the `DemoNet` above and turn it into a running service:
```bash
pip install fastapi uvicorn pydantic
uvicorn deploy.main:app --reload
```
Test with `curl`:
```bash
curl http://127.0.0.1:8000/health
curl -X POST http://127.0.0.1:8000/predict -H "Content-Type: application/json" \
     -d '{"features": [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]}'
```

### Exercise 2 — Image Classification Endpoint
Modify the API to accept a file upload (`UploadFile`) and run a small CNN on it.

### Exercise 3 — Text Sentiment Endpoint
Wrap the LSTM classifier from Module 15 in a FastAPI endpoint.

### Exercise 4 — Dockerize
Build and run the service:
```bash
cd deploy
docker build -t demo-api .
docker run -p 8000:8000 demo-api
```

### Exercise 5 — Batch Endpoint
Add a `/predict_batch` endpoint that takes a list and returns a list of predictions.

### Exercise 6 — Model Versioning
Store the current version in a file and include it in every response.

### Exercise 7 — Add Logging
Log request path, latency, and prediction count per request without logging user data.

### Exercise 8 — Test on Real Cloud
Deploy the Docker image to Hugging Face Spaces, Render, or Fly.io. Get a public URL.

---
## ✅ 16. Deployment Checklist

Before deployment:

- [ ] Model weights saved
- [ ] Model architecture available
- [ ] Preprocessing reproduced
- [ ] `model.eval()` enabled
- [ ] `torch.no_grad()` used
- [ ] Input validation implemented
- [ ] `/health` endpoint available
- [ ] Errors handled
- [ ] Model version tracked
- [ ] Docker image tested
- [ ] CPU/GPU tested
- [ ] Logs available
- [ ] Sensitive information protected

---
## 🎤 17. Interview Questions

1. **How do you deploy a PyTorch model?**  
   Save `state_dict`, package with code + preprocessing, expose via an API (FastAPI), containerize with Docker, deploy to a server.

2. **Why use `model.eval()`?**  
   It disables Dropout and switches BatchNorm to use running statistics.

3. **Why use `torch.no_grad()`?**  
   Prevents building the gradient graph — saves memory and speeds up inference.

4. **What is `state_dict`?**  
   A dictionary mapping each parameter/buffer name to its tensor. Contains **weights only**, not architecture.

5. **How would you expose a model through FastAPI?**  
   Load once at startup, define request/response Pydantic schemas, implement a `/predict` endpoint that runs inference and returns JSON.

6. **Why use Docker for model deployment?**  
   It packages code + dependencies into a reproducible environment — the same container runs anywhere.

7. **CPU vs GPU inference?**  
   CPU is fine for small models; GPUs provide significant speedups for large models but cost more and require availability.

8. **Batch vs real-time inference?**  
   Real-time responds per request; batch processes many samples together for better hardware utilization.

9. **Why must preprocessing remain consistent?**  
   Because the model learned features under specific scaling; different inputs → wrong predictions.

10. **How would you monitor a deployed model?**  
    Track latency, throughput, error rates, prediction distributions, and drift. Include `/health` and structured logs.

---
## ✅ 18. Knowledge Check

- [ ] Save and load models with `state_dict`
- [ ] Explain `model.eval()` and `torch.no_grad()`
- [ ] Reproduce training preprocessing at inference
- [ ] Serve a model via FastAPI
- [ ] Design clean request / response schemas
- [ ] Dockerize a model service
- [ ] Handle CPU vs GPU devices
- [ ] Compare real-time vs batch inference
- [ ] Implement health checks and error handling
- [ ] Add logging without leaking sensitive data
- [ ] Understand ONNX and cloud deployment basics

---
## 🏁 19. Key Takeaways

### The deployment pipeline

```text
Train → Save state_dict → Package → Serve (FastAPI) → Docker → Deploy
```

### The inference essentials

```python
model.eval()
with torch.no_grad():
    prediction = model(x)
```

### The golden rules

> **Preprocessing must match between training and production.**  
> **Load the model once, at startup — not per request.**  
> **Validate inputs early; return generic errors.**  
> **Version your model; log operations; never log secrets.**  
> **Test the container, not just the notebook.**

### 🧭 The Big Picture

> **Deployment is the bridge between Deep Learning and real AI applications.**

---

### 🔗 What's Next?

**Module 17 — Deep Learning Projects**

You now have the **full toolkit**: model design, training, transfer learning, deployment. The final module brings it all together into **end-to-end projects** you can show to employers or use as starting points for your own work.